# DSS150P Final Presentation: Addictive Usage & Well-being Risk Detection

**Team Members:**
- QUERIJERO, ELIJAH BRADLEY
- AGUAVIVA, YUVAL MA. EZEKIEL
- MALICDEM, VINCE MARTIN

---

### The Problem
Short-video addiction poses a serious mental health risk, especially for students and young adults. We built an automated Data Engineering pipeline to process millions of messy interaction events into clean, daily behavioral profiles. 

By detecting compulsive usage signatures (e.g., watching at 2–4 AM, high volume, and high "hate" reactions signaling negative emotional response), our pipeline lays the groundwork for digital well-being nudges.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add the scripts folder to Python's path so we can import our pipeline config
sys.path.append('scripts')
from config import CURATED_DATA_DIR, SOURCE_DATA_DIR

# Set visualization styles for the presentation
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

## 1. The Raw Data (The Mess)
First, let's look at the raw data provided by the Tsinghua FIB Lab. It's stored in CSV format, where every row is a single tag exposure. It is full of duplicates and not ready for analysis.

In [ ]:
# Load a sample of the raw data (using our pipeline's configuration path)
raw_csv_path = SOURCE_DATA_DIR / "interaction_sampled.csv"

# Read just the first 1000 rows for the demo
if raw_csv_path.exists():
    raw_df = pd.read_csv(raw_csv_path, nrows=1000)
    display(raw_df.head())
else:
    print(f"Source file not found at {raw_csv_path}")

## 2. The Curated Layer (The Solution)
Our pipeline cleans this data, drops exact duplicates, builds user sessions, and outputs clean **Parquet** partitions. Let's dynamically load all partitions of the curated `daily_user_features` table.

In [ ]:
# Locate the curated features directory
features_dir = CURATED_DATA_DIR / "daily_user_features"

# Load ALL partitions (e.g. p_date=20220916, 20220917, etc.) into a single DataFrame
if features_dir.exists():
    frames = [pd.read_parquet(p) for p in features_dir.glob("p_date=*/*.parquet")]
    if frames:
        features_df = pd.concat(frames, ignore_index=True)
        print(f"Successfully loaded {len(features_df)} daily user profiles across all partitions!")
        display(features_df.head())
    else:
        print("No feature partitions found. Please run the pipeline first!")
else:
    print("Curated data directory not found. Please run the pipeline first!")

## 3. Well-Being Risk Analytics (Bonus +10)
Now we apply our risk-scoring logic to the clean data to flag compulsive users.

**Risk Flags:**
1. Top 25% in total watch time.
2. High Hate Rate (> 1% of interactions are "hates").
3. Late Night Binging (> 10% of watch time between 2 AM and 4 AM).

In [ ]:
if 'features_df' in locals():
    # Calculate risk scores using the same thresholds from our analytics script
    features_df['high_volume'] = (features_df['total_watch_seconds'] > features_df['total_watch_seconds'].quantile(0.75)).astype(int)
    features_df['high_hate'] = (features_df['hate_rate'] > 0.01).astype(int)
    features_df['late_night_binge'] = (features_df['late_night_share'] > 0.1).astype(int)
    
    features_df['risk_score'] = features_df['high_volume'] + features_df['high_hate'] + features_df['late_night_binge']
    
    # Isolate at-risk users (score >= 1 for this small sample dataset)
    high_risk_users = features_df[features_df['risk_score'] >= 1]
    
    print(f"Total Users Analyzed: {len(features_df)}")
    print(f"At-Risk Users Detected: {len(high_risk_users)}")
    display(high_risk_users[['user_id', 'total_watch_seconds', 'hate_rate', 'late_night_share', 'risk_score']].head())

## 4. Visualizing the Compulsion
Let's visualize the difference between normal users and at-risk users based on their Late Night Share and Total Watch Time.

In [ ]:
if 'features_df' in locals():
    plt.figure(figsize=(10, 6))
    
    # Create a color mapping for the risk score
    scatter = plt.scatter(x=features_df['late_night_share'], 
                          y=features_df['total_watch_seconds'], 
                          c=features_df['risk_score'], 
                          cmap='coolwarm', 
                          alpha=0.7, 
                          s=50)
    
    plt.colorbar(scatter, label='Risk Score (0 to 3)')
    plt.title('User Well-Being Risk Analysis: Watch Volume vs. Late Night Binging', fontsize=14)
    plt.xlabel('Late Night Watch Share (2 AM - 4 AM)', fontsize=12)
    plt.ylabel('Total Watch Time (Seconds)', fontsize=12)
    
    # Add a subtle threshold line for late night binging (10%)
    plt.axvline(x=0.1, color='red', linestyle='--', alpha=0.3, label='Late Night Threshold')
    plt.legend()
    plt.tight_layout()
    plt.show()

### Conclusion
Our data engineering pipeline successfully structures the massive volume of interaction events, making it possible to instantly isolate at-risk users. This clean data model is now ready to be plugged into a production alerting system for immediate digital well-being interventions.